# 06 — Evaluation, scaling, and reproducibility

A VLM benchmark is a measurement protocol, not an intrinsic model property. Compare identical dataset versions, split, prompt, image resize/crop/tiling, number of images, decoding settings, and answer normalization. Track task slices (OCR, chart, grounding, reasoning, video) because a mean can hide a serious weakness. Include latency, memory, throughput, and cost on the hardware you will use.

Typical evaluations: retrieval Recall@K; VQA accuracy; OCR exact/normalized match; document/chart QA; phrase grounding localization; video temporal reasoning; calibration/selective prediction; robustness under blur, lighting, distractors, and distribution shift. For robotics, add closed-loop success, collision/contact safety, recovery, and latency under real camera conditions.

In [ ]:
import torch
import numpy as np
import pandas as pd

# Tiny retrieval evaluation from a similarity matrix; rows are queries, columns are candidates.
scores = torch.tensor([[0.9, 0.2, 0.1], [0.1, 0.8, 0.2], [0.2, 0.3, 0.7]])
correct = torch.arange(scores.shape[0])
ranking = scores.argsort(dim=1, descending=True)
recall_at_1 = (ranking[:, :1] == correct[:, None]).any(dim=1).float().mean()
recall_at_2 = (ranking[:, :2] == correct[:, None]).any(dim=1).float().mean()
print('R@1:', float(recall_at_1), 'R@2:', float(recall_at_2))

# A benchmark ledger makes protocol differences visible instead of hiding them in one score.
ledger = pd.DataFrame([
    {'slice': 'OCR', 'metric': 'normalized exact match', 'n': 0, 'prompt': 'read all visible text', 'status': 'not run'},
    {'slice': 'grounding', 'metric': 'IoU@0.5', 'n': 0, 'prompt': 'box the target', 'status': 'not run'},
    {'slice': 'latency', 'metric': 'p95 milliseconds', 'n': 0, 'prompt': 'fixed prompt', 'status': 'not run'},
])
ledger

## Scaling laws are conditional

More parameters or tokens do not guarantee better downstream performance. Data quality/diversity, compute allocation, architecture, optimization, and evaluation contamination matter. Track training curves against tokens and compute, not only epochs. Watch train/validation gap, loss spikes, duplicate rates, gradient norms, batch token counts, and modality balance. For high-resolution images, the visual token count can dominate attention costs; image tiling and adaptive resolution trade detail against compute/context budget.

## Reproducible experiment record

Record: source commit; Python/PyTorch versions; seed and deterministic settings; exact data manifest hash and license; tokenizer; image processor and resolution; architecture/config; optimizer/scheduler; effective batch (including accumulation); steps/tokens; checkpoint selection rule; hardware; evaluation prompts and scripts. Repeat across seeds where feasible. Store per-example errors for review, respecting privacy and data agreements.

## Ablation questions

- Does image conditioning matter? Shuffle image-caption pairs at validation.
- Does OCR need higher resolution? Compare fixed compute budgets and disclose resize policy.
- Is a gain from data or architecture? Hold compute and data constant.
- Does fine-tuning damage general capability? Re-run a held-out general evaluation.
- Does offline accuracy translate to robot success? Evaluate feedback, latency, contact, and recovery in closed loop.